# 📔 Journal de bord — Détection précoce du décrochage étudiant en L1

*Michael Staudt — certification « Concevoir et implémenter une solution d'IA »*

**Dépôt public du projet : <https://github.com/MichaeLab34/examen>**

---

Le rendu propre, avec l'histoire dans le bon ordre et les figures, est dans
`notebooks/decrochage_etudiant.ipynb`. Ici je note au jour le jour ce que j'ai
fait, ce qui a coincé, et pourquoi j'ai tranché dans un sens plutôt qu'un autre.
C'est ce que je relis avant l'oral : le jury ne demande pas le code, il demande le
pourquoi des choix.


## 📅 Journal de bord – Jour 1 (08/07/2026)

### 🔧 Étapes / Actions réalisées

- Découverte du sujet et lecture de l'énoncé,
- Structuration des dossiers et mise en place du suivi **git**,
- Chargement des 3 CSV (`utf-8-sig`),
- Premier modèle rapide sur toutes les colonnes, sans sélection préalable, juste pour
  voir ce que ça donne : **AUC > 0,95 dès le premier essai**.

### 📝 Observations / Difficultés

Un score aussi haut au premier essai n'est pas normal. À mi-S1, on ne prédit pas le
décrochage aussi bien. Soit les données sont très faciles, soit une colonne
interdite est entrée dans le modèle.

En regardant les importances : `moyenne_finale` et `moyenne_partiels_s1` pèsent
beaucoup plus que toutes les autres. Or `moyenne_finale` est le résultat de fin
d'année : je prédis le passé avec le futur.

**Note** : la bonne question n'est pas « est-ce que la colonne est corrélée à
l'abandon ? » mais « est-ce que j'aurais cette valeur le jour de la décision, à
mi-S1 ? ». Une colonne qui prédit trop bien est un suspect.

### ✅ Prochaines étapes
- Reprendre les 33 colonnes une par une avec ce critère,
- Séparer ce qui est réellement disponible à mi-S1 du reste.


## 📅 Journal de bord – Jour 2 (10/07/2026)

### 🔧 Étapes / Actions réalisées

- Tri des colonnes suspectes en deux familles :
  - `moyenne_finale` → **fuite de données** : c'est une partie de la réponse,
  - `moyenne_partiels_s1`, `nb_ue_validees_s1` → **fuite temporelle**,
- Contrôle des trois leurres annoncés par l'énoncé (`groupe_td`,
  `couleur_carte_etudiante`, `jour_inscription`) : taux d'abandon tracé par
  modalité.

### 📝 Observations / Difficultés

Le deuxième cas m'a bloqué un moment : des partiels de S1 ont l'air légitimes
puisqu'on est en S1. Ce qui tranche, c'est le *quand* — ces colonnes sont
consolidées en fin de S1, alors que le score est calculé à mi-S1. En production,
elles seront vides.

Pour les leurres, je n'ai pas voulu croire l'énoncé sur parole. Le taux d'abandon
par modalité est plat, environ 2 points d'écart-type. Je ne l'affirme pas, je le
montre : un leurre qui serait corrélé serait soit une erreur de l'énoncé, soit une
fuite cachée.

**Note** : deux problèmes différents (données / temporel), un seul symptôme (« ça
marche trop bien »). Chaque exclusion est donc justifiée dans le code, à côté de la
règle qui l'applique : le motif doit rester lisible par quiconque reprendra le
périmètre.

### ✅ Prochaines étapes
- Fixer la liste des colonnes autorisées dans le code, pas dans un commentaire,
- Ajouter un contrôle qui échoue si une colonne interdite revient.


## 📅 Journal de bord – Jour 3 (11/07/2026)

### 🔧 Étapes / Actions réalisées

- Liste des colonnes autorisées centralisée dans `features.py`, une seule fonction
  de référence : `scoring_feature_columns()`,
- Ajout du garde-fou `assert_no_leakage()` : le pipeline s'arrête si une colonne
  interdite est présente,
- Réentraînement après verrouillage : il reste **31 features**, l'AUC reste autour
  de **0,95**.

### 📝 Observations / Difficultés

Une fuite ne laisse aucun symptôme visible : le modèle tourne, les métriques sont
bonnes. Rien n'empêche de la réintroduire à la prochaine évolution du périmètre. Un
commentaire dans le code ne protège de rien, d'où la fonction unique et le contrôle
qui plante.

L'AUC toujours à 0,95 après nettoyage m'a fait douter : est-ce que j'ai raté une
fuite ? J'ai réaudité les colonnes une par une, il ne reste rien d'interdit. Les
données sont synthétiques et le signal est fort : une AUC élevée est cohérente ici.

**Note (pour l'oral)** : si on me demande « 95 % d'AUC, il n'y a pas une fuite ? »,
je réponds avec le périmètre verrouillé, le garde-fou et la nature synthétique des
données.

### ✅ Prochaines étapes
- Écrire le nettoyage dans un module réutilisable, pas dans le notebook,
- Décider où placer l'imputation des valeurs manquantes.


## 📅 Journal de bord – Jour 4 (15/07/2026)

### 🔧 Étapes / Actions réalisées

- Nettoyage écrit dans `preprocessing.clean_raw`, dans un module et pas dans le
  notebook, parce qu'il devra être rejoué à l'identique en production :
  - nombres au format français : `"85,0"` → `85.0`, `"14.4 km"` → `14.4`, `"61.4%"` → `61.4`,
  - dates dans 3 formats différents, harmonisées,
  - ~40 doublons retirés, on retombe sur **5 200 lignes**,
- Imputation commencée puis mise en attente.

### 📝 Observations / Difficultés

J'ai commencé à remplir les valeurs manquantes dans le nettoyage, puis j'ai arrêté :
calculer la médiane sur tout le jeu fait passer de l'information du test vers le
train. La statistique d'imputation doit s'apprendre sur le train seulement, donc sa
place est **dans la Pipeline**, pas dans le nettoyage.

Deux options pour les manquants :
- **remplacement par la médiane** : garde toutes les lignes, donc la
  représentativité — utile car `abandon` est la classe minoritaire (~28 %),
- **suppression des lignes** : plus propre en apparence, mais risque de supprimer
  surtout la classe qui m'intéresse.

Je pars sur la médiane. Un imputeur régressif (estimer une colonne à partir des
autres) serait plus fin, mais plus complexe et plus exposé à la fuite : gardé pour
une v2, avec un audit de fuite obligatoire.

### ✅ Prochaines étapes
- Comparer plusieurs algorithmes sur la même Pipeline,
- Commencer par une baseline pour savoir ce que vaut vraiment un score.


## 📅 Journal de bord – Jour 5 (17/07/2026)

### 🔧 Étapes / Actions réalisées

- Baseline avec `DummyClassifier` (AUC ≈ 0,5) pour connaître le plancher,
- Comparaison sur la **même** Pipeline : régression logistique, Random Forest,
  XGBoost — courbes ROC superposées et tableau AUC / AP / F1 / rappel,
- Gestion du déséquilibre des classes par `class_weight`.

### 📝 Observations / Difficultés

Mon réflexe de départ était « sur du tabulaire, XGBoost gagne ». Ici, il n'apporte
aucun gain d'AUC significatif face à la régression logistique.

Le métier demande de pouvoir expliquer à un référent pourquoi tel étudiant est
signalé. Les coefficients d'une régression logistique se lisent, ceux d'un boosting
non. Je choisis donc la régression logistique : l'explicabilité contre un gain de
performance faible et non significatif. C'est en plus le modèle le moins coûteux à
entraîner.

**Note** : le boosting n'est pas abandonné, je le garde comme candidat pour un futur
A/B test — mais pas comme modèle livré.

### ✅ Prochaines étapes
- Régler les hyperparamètres du modèle retenu,
- Choisir le seuil de décision.


## 📅 Journal de bord – Jour 6 (18/07/2026)

### 🔧 Étapes / Actions réalisées

- `GridSearchCV` sur `C`, validation croisée stratifiée, scoring AUC, **sur le train
  seulement** ; AUC en validation croisée comparée à l'AUC de test → pas de
  surapprentissage,
- Choix du seuil de décision à partir du coût métier.

### 📝 Observations / Difficultés

J'ai failli garder **0,5** parce que c'est la valeur par défaut. Mais 0,5 revient à
dire qu'un faux négatif et un faux positif coûtent la même chose. Rater un
décrocheur peut lui coûter son année ; une fausse alerte coûte environ 20 minutes
d'entretien à un référent. Ne pas choisir, c'est quand même choisir.

J'ai donc écrit le coût explicitement : **FN:FP = 5:1**, et je retiens le seuil qui
minimise le coût total, calculé **sur la validation, jamais sur le test**. L'optimum
tombe autour de **0,3** et le rappel monte à **90-96 %**, ce qui correspond à
l'objectif : mieux vaut convoquer un étudiant pour rien que d'en laisser passer un.

**Note** : le ratio 5:1 est une hypothèse métier, pas une vérité. Il se rediscute
avec la direction et je peux relancer le calcul avec d'autres valeurs. Le test, lui,
reste intact, sinon il ne mesure plus rien.

### ✅ Prochaines étapes
- Sortir la logique du notebook vers `src/decrochage/`,
- Traiter le RGPD et l'équité.


## 📅 Journal de bord – Jour 7 (22/07/2026)

### 🔧 Étapes / Actions réalisées

- Logique déplacée dans `src/decrochage/` : le notebook, la CLI et l'API importent
  le même nettoyage et le même périmètre de colonnes,
- Bundle joblib qui embarque le catalogue des formations, pour ne dépendre d'aucun
  fichier annexe au moment de servir,
- Découpage Bronze / Silver / Gold et pseudonymisation des identifiants,
- Audit d'équité par sous-groupes.

### 📝 Observations / Difficultés

Trois implémentations « équivalentes » du nettoyage finissent toujours par diverger.
D'où le module unique, importé partout.

Côté RGPD, traité pendant la conception et pas à la fin :
- **Bronze** : données brutes avec les données personnelles, isolable, auditable,
  purgeable,
- **dès Silver** : identifiants pseudonymisés en HMAC-SHA-256, la suite du pipeline
  ne voit jamais un identifiant réel.

La protection est dans le code, pas dans un document.

Sur l'équité : retirer `sexe` ne suffit pas, le modèle peut discriminer par des
variables corrélées (origine, statut boursier). J'ai donc ajouté un audit par
sous-groupes, avec un écart de rappel maximum de **10 points**. Et l'humain reste
dans la boucle, dans l'esprit de l'article 22 : le score propose, le référent
décide. Pas de décision automatique sur ce sujet.

### ✅ Prochaines étapes
- Rejouer le pipeline complet de bout en bout,
- Reprendre le recul général et lister ce qui reste à améliorer.


## 📅 Journal de bord – Jour 8 (23/07/2026)

### 🔧 Étapes / Actions réalisées

- Schéma de l'architecture cible : ingestion SI/LMS → Bronze → Silver → Gold →
  entraînement ou scoring → exposition → restitution aux référents,
- Tableau des contraintes techniques, réglementaires, organisationnelles et
  économiques, avec la réponse retenue pour chacune,
- Mesure de performance : indicateurs techniques (AUC, rappel, précision, F1,
  seuil) **et** métier (étudiants détectés, coût d'un faux négatif),
- Explicabilité : coefficients, importance des variables et SHAP,
- Audit d'équité par sous-groupes : écart de rappel mesuré à **1,9 point** sur
  `sexe` et `boursier` ; `etablissement_origine` est observé mais ses modalités
  les plus petites (31 et 74 étudiants) sont trop peu peuplées pour porter un seuil,
- Cible secondaire `moyenne_finale` en régression : Ridge et Random Forest,
  R² ≈ 0,68, MAE ≈ 2,3 points sur 20.

### 📝 Observations / Difficultés

Sur le dimensionnement, j'ai commencé par réfléchir en « architecture idéale » avant
de revenir à la réalité du volume : ~5 200 étudiants par an, un scoring par semestre
et des contrôles de dérive hebdomadaires. Kubernetes n'a aucun sens ici, un VPS
européen conteneurisé suffit. Le sur-dimensionnement est un vrai risque à l'oral : il
coûte cher et ne se justifie par aucun chiffre du projet.

Sur les impacts, j'ai failli traduire l'AUC en euros économisés. Je ne l'ai pas
fait : rien dans les données ne permet de mesurer l'effet d'un accompagnement. Je
donne donc le nombre d'étudiants détectés et le coût d'un faux négatif, et je renvoie
le ROI à un pilote mesuré.

Sur la régression, R² ≈ 0,68 et une erreur d'environ 2 points sur 20 : c'est assez
pour trier entre soutien léger et soutien renforcé, pas pour annoncer une note. Je le
dis explicitement, sinon quelqu'un finira par afficher la note prédite à un étudiant.

### ✅ Prochaines étapes
- Mettre en place le suivi en exploitation : dérive, réentraînement, gestion des
  versions,
- Vérifier que tout tourne hors notebook, dans la stack Docker.


## 📅 Journal de bord – Jour 9 (24/07/2026)

### 🔧 Étapes / Actions réalisées

- Monitoring de dérive exécutable : `decrochage drift-report`, PSI par variable
  numérique, niveaux `ok` / `watch` / `alert`, rapport persisté en Gold,
- Politique de réentraînement écrite dans `operations.py` : contrôle à chaque batch,
  réentraînement annuel, anticipé seulement si dérive **et** labels récents,
- Suivi MLflow : un run par entraînement (paramètres, métriques, bundle en
  artefact), alias `candidate` / `production` / `archived` pour la promotion et
  le rollback,
- Barrière de promotion chiffrée : AUC ≥ 0,85, rappel ≥ 0,90 sans régression, écart de
  rappel ≤ 10 points, puis validation humaine,
- Stack Docker : API derrière Caddy, `/metrics` collecté par Prometheus, tableau de bord
  Grafana, deux règles d'alerte, APScheduler pour les tâches planifiées,
- Captures des services en fonctionnement, prises avec Playwright, versées au
  notebook comme preuves d'exploitation.

### 📝 Observations / Difficultés

Premier réflexe : réentraîner tous les mois. Ça ne tient pas ici. La cible `abandon`
n'est connue qu'une fois la cohorte terminée : réentraîner mensuellement reviendrait
à réapprendre sur des étiquettes qui n'existent pas encore. La dérive déclenche donc
une **enquête**, pas un réentraînement automatique.

Deuxième point, l'alerting : sans garde-fou, un seuil dépassé génère une alerte par
minute. J'ai ajouté une temporisation et une hystérésis, plus un heartbeat pour
détecter le cas inverse — une tâche silencieuse qui a cessé de tourner sans que
personne ne le voie.

Troisième point : tout ce qui a besoin d'un processus réseau (Prometheus, Grafana,
MLflow, ordonnanceur) ne peut pas être « prouvé » depuis une cellule de notebook. Je
l'ai donc validé dans Docker et rapatrié les captures, plutôt que de simuler des
résultats dans le notebook.

**Note** : les journaux applicatifs ne contiennent ni identifiant ni payload étudiant
— seulement route, statut, durée et `X-Request-ID`. Un log est une base de données
qui s'ignore, il doit respecter la même minimisation que le reste.

### ✅ Prochaines étapes
- Relire l'ensemble à froid et faire le bilan,
- Préparer les réponses aux questions attendues du jury.


## 📅 Journal de bord – Jour 10 (27/07/2026)

En relisant l'ensemble à froid pour préparer le bilan, un point m'a sauté aux yeux :
l'éco-conception, je l'**affirmais sans l'avoir mesurée**. Je reprends donc le dossier
là-dessus avant de conclure.

### 🔧 Étapes / Actions réalisées

- Module `src/decrochage/ecodesign.py` : mesure de la durée, de l'énergie et de
  l'empreinte d'un entraînement via **CodeCarbon** (mix électrique français), avec
  dégradation propre si la librairie est absente,
- Mesure branchée **dans la boucle de comparaison des modèles** (§8.3), donc au
  moment même où chaque candidat est entraîné,
- Tableau d'arbitrage : surcoût de calcul et **coût par point d'AUC** face au modèle
  retenu,
- Résultat mesuré : Random Forest coûte **≈ 23 ×** le temps de la régression
  logistique pour une AUC **inférieure** ; XGBoost **≈ 12 ×**, pour une AUC
  inférieure elle aussi. Coût par point d'AUC = **infini** dans les deux cas,
- Recensement des six leviers d'éco-conception du projet, classés par impact réel.

### 📝 Observations / Difficultés

Jusqu'ici j'écrivais « la régression logistique est plus sobre » — c'était une
intuition raisonnable, pas un fait établi. Maintenant je peux donner le rapport de
coût. C'est la même exigence que sur les leurres : ne pas affirmer, montrer.

Le point le plus intéressant est ailleurs. Les valeurs absolues sont dérisoires
(**0,022 Wh** pour toute la comparaison) : sur 5 200 lignes, le choix de
l'algorithme ne sauve pas la planète. Le levier réel, c'est la **fréquence de
réentraînement** — passer de mensuel à annuel supprime onze entraînements par an —
puis le dimensionnement de l'infrastructure. Je le présente dans cet ordre, sinon
l'éco-conception devient un argument décoratif.

**Difficulté rencontrée** : la mesure ne remontait rien dans le notebook alors
qu'elle fonctionnait en ligne de commande. Cause trouvée : le notebook déclarait le
kernel **`indusense-venv`**, l'environnement d'un autre projet. Il ne tournait donc
pas dans son propre `.venv` — un livrable qu'on croit reproductible et qui ne l'est
pas. Kernel corrigé, mesure obtenue.

**Note** : limites à annoncer à l'oral — sous Windows l'interface RAPL est
indisponible, l'estimation CPU est approximative et dépend de la machine. Ces
chiffres servent à **comparer les modèles entre eux**, pas à publier une empreinte
absolue.

### ✅ Prochaines étapes
- Répercuter l'éco-conception dans le support de soutenance et l'architecture,
- Vérifier que le notebook s'exécute de bout en bout dans le `.venv` du projet,
- Écrire le bilan, maintenant que plus rien n'est affirmé sans preuve.


## 📅 Journal de bord – Jour 11 (28/07/2026)

Dernière journée avant l'oral. Je ne code plus : je relis le dossier dans les
conditions du jury, c'est-à-dire sans le contexte que j'ai en tête.

### 🔧 Étapes / Actions réalisées

- Reprise du choix d'hébergement : l'option retenue devient le **serveur qui héberge
  déjà le LMS**, le VPS européen conteneurisé passant en solution de repli. Répercuté
  dans `ARCHITECTURE_PROJET.md`, `docs/run_architecture.md`, le notebook (§13.3) et
  le support de soutenance,
- Passage de toute la documentation en français : cinq documents de `docs/` étaient
  rédigés en anglais (fiche modèle, plan de surveillance, modèle de menaces,
  redevabilité RGPD, industrialisation),
- Mise à jour de `ARCHITECTURE_PROJET.md`, qui décrivait **8 modules sur 17** et
  **5 commandes CLI sur 15** : tout le volet exploitation et l'éco-conception y
  manquaient,
- Relecture croisée notebook / journal / support / slides, chiffre par chiffre.

### 📝 Observations / Difficultés

La relecture croisée fait apparaître quatre écarts de cohérence entre documents.

Le plus important porte sur l'**audit d'équité, dans le support et les slides** : la
fourchette de rappel par sous-groupe y était donnée « de 0,935 à 0,975 » pour un
écart maximal de 1,9 point. Ces deux valeurs sont incompatibles — 0,975 moins 0,935
fait 4 points. Les valeurs exactes, relues dans la sortie du notebook, sont **0,946 à
0,966**.

Le même passage rangeait `etablissement_origine` parmi les sous-groupes couverts par
cet écart. Le tableau du notebook montre pour cette variable un rappel de **0,667**
sur un groupe de 31 étudiants : l'écart de 1,9 point ne la couvre donc pas. Le
périmètre est désormais explicite, et il tient méthodologiquement — à 31 et 74
étudiants, un rappel se déplace de plusieurs points dès qu'un seul étudiant change de
côté. Je pilote l'alerte d'équité sur `sexe` et `boursier`, et j'observe l'origine
scolaire sans en faire un seuil bloquant.

Les trois autres écarts sont dans le notebook : une table `gold_student_feature` qui
n'existe pas — c'est `gold_training_feature` ; un « batch hebdomadaire » annoncé en
§11.2 alors que la §8.3 retient un batch par semestre, la fréquence hebdomadaire étant
celle des contrôles de dérive et non du scoring ; et un renvoi à la §15 au lieu de la
§11 pour le dimensionnement de l'infrastructure.

**Note** : les quatre ont la même origine — un chiffre ou un nom recopié d'un document
à l'autre plutôt que relu à la source. Le notebook est la source de vérité ; tout ce
qui le cite doit en découler. C'est la même exigence que le verrou anti-fuite,
appliquée à la documentation.

Sur l'hébergement, j'avais raisonné en « quoi louer » sans poser la question
préalable : de quoi l'université dispose-t-elle déjà ? Les données d'engagement
**viennent** du LMS. Les traiter sur la machine qui l'héberge évite tout transfert
hors du système d'information de l'université, tout sous-traitant supplémentaire au
sens du RGPD, et tout coût d'hébergement. C'est aussi le levier n° 3 de
l'éco-conception vue hier : pas de machine supplémentaire à alimenter. La condition
est le cloisonnement — le scoring tourne dans son propre conteneur, avec des
ressources plafonnées, pour qu'un
entraînement ne dégrade jamais le LMS en période de partiels. Le VPS reste le repli si
la DSI préfère isoler le service, et la conteneurisation fait que ce choix ne m'engage
pas : la même image tourne dans les deux cas.

### ✅ Prochaines étapes
- Dérouler la présentation à voix haute, chronomètre en main,
- Relire le tableau des questions du jury,
- Ne plus rien changer au dossier.


## 📅 Journal de bord – Jour 12 (29/07/2026)

Le jour 11 se terminait par « ne plus rien changer au dossier ». Je le rouvre quand
même, pour une raison précise : mon diagramme d'architecture comportait un bloc
« Restitution & pilotage (décision HUMAINE) » qui n'avait **aucune implémentation**.
Tant qu'il n'existait pas, ma phrase « le score propose, l'humain décide » restait
une intention — aucun référent n'avait matériellement les moyens de décider. Le seul
chemin de sortie du modèle était un `POST /predict` qui renvoie une probabilité, sans
identité, sans explication et sans trace de consultation.

### 🔧 Étapes / Actions réalisées

- **Portail de restitution** (`src/decrochage/portal/`) : router FastAPI monté dans le
  service existant, rendu serveur Jinja2, aucun build JavaScript, aucune ressource
  distante. Désactivé par défaut, refuse de démarrer sans secret de session,
- Trois rôles cloisonnés — `referent` (cohorte de ses filières, fiche explicable,
  export), `pilote` (indicateurs agrégés, simulateur de seuil), `auditeur` (journal
  des consultations, rétention) — le périmètre étant appliqué **dans la requête SQL**,
  jamais dans le gabarit,
- **Explicabilité analytique** : sur la régression logistique, la contribution d'une
  variable au log-odds est exactement `coefficient × valeur transformée`. Pas de SHAP
  à l'exécution ; un test vérifie que la somme des contributions retombe sur la
  décision du modèle,
- **52 tests** dédiés (authentification, confidentialité, vues, explicabilité), soit
  **112** au total,
- Mise en service réelle dans la stack Docker derrière Caddy en HTTPS, et trois démos
  filmées, une par profil, avec leur prompt rejouable,
- **Estampillage du modèle** dans `training.py` : `trained_at` et `model_version`.

### 📝 Observations / Difficultés

La session de développement a été interrompue en cours de revue. En la reprenant, le
journal de validation annonçait « 109 tests verts, lint et format conformes ». En
rejouant la suite : **21 tests en échec**, `ruff` et `black` en échec également. Le
rapport décrivait un état qui avait cessé d'être vrai — la revue avait appliqué ses
correctifs sans rejouer les tests derrière.

C'est exactement le fil du projet, appliqué à moi-même : **un rapport de validation
n'est pas une preuve, seule l'exécution en est une**. J'ai passé le jour 10 à mesurer
l'éco-conception que j'affirmais, et le jour 11 à traquer des chiffres recopiés au
lieu d'être relus. Ici, j'ai failli faire confiance à un compte-rendu plutôt qu'à la
commande.

Les défauts trouvés ont tous la même forme : **un durcissement appliqué à moitié**.
Une colonne de probabilité maximale avait été retirée du modèle de données — à raison,
le maximum d'un groupe est le score d'un étudiant précis — mais le gabarit qui
l'affichait n'avait pas suivi : la vue de pilotage était entièrement cassée. Le
cookie de session était passé en `Secure` par défaut, sans que les tests, qui parlent
en HTTP, en soient informés : chaque test authentifié se rejouait en anonyme sans le
dire. Et l'export avait été restreint au seul référent, alors que ma matrice de droits
l'ouvre aussi au pilote.

Deux tests étaient faux, et c'est le cas le plus instructif. L'un vérifiait les
colonnes du CSV exporté en ignorant que le fichier commence par un BOM UTF-8 — le code
était juste, l'assertion passait à côté. L'autre s'appelait « historique entre lots »
mais simulait deux lots en écrivant **deux fois dans le même lot** ; la déduplication
introduite entre-temps rendait cela correctement invisible. Dans les deux cas la
tentation était de relâcher l'assertion pour repasser au vert. J'ai corrigé les tests
pour qu'ils testent ce que leur nom annonce, et j'en ai ajouté deux.

Enfin, le portail a joué un rôle que je n'attendais pas : celui de **révélateur**. Son
bandeau de contexte affiche le lot, la date, le seuil et la version de modèle. Il
affichait « modèle non renseigné ». La cause n'était pas dans le portail :
`train_model` construisait ses métadonnées à partir des seules métriques, sans jamais
estampiller le bundle. La colonne `model_version` de `gold_prediction` restait donc
vide depuis le début, et personne ne l'avait vu — parce que rien, jusqu'ici, ne
l'affichait. Une décision qu'on ne peut pas rattacher à un modèle n'est pas auditable.
Le préfixe `local-` est délibéré : il distingue une construction locale d'une version
promue dans le registre MLflow, qui est numérique.

**Note** : construire l'écran qui restitue une information est le meilleur moyen de
découvrir que cette information n'existait pas. C'est un argument à tenir à l'oral :
le portail n'est pas une couche cosmétique ajoutée à la fin, il a corrigé un défaut
d'auditabilité au cœur de la chaîne.

### ✅ Prochaines étapes
- Rejouer la démonstration complète — `medallion-load`, `train`, `predict`, portail —
  sur une base vierge, pour vérifier que la procédure de mise en service tient,
- Intégrer le slide « Restitution aux référents » au conducteur et recaler le
  minutage,
- Revenir à la consigne du jour 11 : ne plus rien changer.


## 🎯 Bilan — ce que je retiens (29/07/2026)

Un seul fil sur tout le projet : la validité avant la performance. Le verrou
anti-fuite, la séparation validation / test, le nettoyage partagé, la
pseudonymisation dès Silver, la barrière de promotion chiffrée — c'est toujours la
même idée : faire en sorte que le chiffre final veuille dire quelque chose.

**À refaire pareil**
- Mettre les décisions à risque (fuite, périmètre de colonnes) dans des fonctions
  testées, pas dans des cellules de notebook,
- Écrire les valeurs métier (coût 5:1, seuils d'alerte, barrière de promotion)
  explicitement plutôt que de les subir,
- Traiter le RGPD et l'éthique dans l'architecture, pas dans un document de fin,
- Dimensionner selon le volume réel plutôt que selon l'état de l'art,
- **Mesurer ce que j'avance** — le jour 10 l'a rappelé : « le modèle est sobre »
  n'était qu'une intuition tant que le coût de calcul n'était pas chiffré,
- **Faire découler les documents d'une source unique** — le jour 11 l'a montré : les
  quatre écarts trouvés venaient tous d'un chiffre recopié au lieu d'être relu dans le
  notebook,
- **Rejouer plutôt que lire un rapport** — le jour 12 l'a montré : un compte-rendu
  annonçait 109 tests verts, l'exécution en donnait 21 rouges. Un état validé se
  périme dès qu'on modifie quelque chose derrière.

**À creuser avec plus de temps**
- Imputeur régressif, avec son audit de fuite,
- A/B test boosting contre régression logistique, pour mesurer si le gain vaut la
  perte d'explicabilité,
- Étude d'équité plus fine que le seul écart de rappel,
- **Boucle de retour des référents** : enregistrer l'issue du contact (entretien
  réalisé, dispositif engagé, abandon confirmé) produirait les labels récents que ma
  politique de réentraînement suppose sans que rien ne les fabrique aujourd'hui.
  C'est le prolongement naturel du portail, et le plus utile,
- Effet réel de l'accompagnement déclenché : c'est la seule mesure qui compte
  vraiment, et aucune donnée du projet ne permet de l'estimer aujourd'hui.

**Note** : tout ceci vaut sur des **données synthétiques**. Avant un déploiement
réel, il faut revalider sur des données réelles et passer par un A/B test pour
mesurer l'effet réel du dispositif. Une bonne AUC ne prouve pas que ça aide les
étudiants.
